In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# 1. CARREGAMENTO DOS DADOS
# ============================================================

rel = pd.read_excel("Relacionamentos.xlsx")
rel = rel[["ID_ALUNO", "Tema Relacionamento", "Motivo do Contato"]].fillna("Não informado")


# ============================================================
# 2. TEXTO DE CADA REGISTRO HISTÓRICO
# ============================================================

rel["texto"] = (
    rel["Motivo do Contato"].astype(str) + " " + rel["Tema Relacionamento"].astype(str)
)


# ============================================================
# 3. VETORIZAÇÃO DO HISTÓRICO (TF-IDF)
# ============================================================
# TF-IDF em vez de contagem simples: palavras que aparecem em
# muitos registros (ex: "via", "do", "de") pesam menos, e palavras
# específicas de um assunto (ex: "boleto", "trancamento") pesam mais.
# Isso evita que conectores comuns dominem a similaridade.

stopwords_pt = [
    "de", "do", "da", "das", "dos", "a", "o", "as", "os", "um", "uma",
    "uns", "umas", "para", "com", "em", "no", "na", "nos", "nas", "via",
    "ao", "aos", "à", "às", "e", "ou", "que",
]

vectorizer = TfidfVectorizer(stop_words=stopwords_pt)
X = vectorizer.fit_transform(rel["texto"])

print("Matriz de registros históricos:", X.shape)


# ============================================================
# 4. ENTRADA: PERGUNTA DO ESTUDANTE
# ============================================================

pergunta = input("Digite a pergunta do estudante: ")
# exemplo pra testar: "segunda via do boleto"


# ============================================================
# 5. REPRESENTAÇÃO EM VETOR DA PERGUNTA
# ============================================================
# Mesmo espaço vetorial (mesmo vocabulário e pesos) do histórico.

vetor_pergunta = vectorizer.transform([pergunta])


# ============================================================
# 6. ÁLGEBRA LINEAR: PRODUTO ESCALAR E SIMILARIDADE DE COSSENO
# ============================================================

# cálculo manual, pra deixar explícita a fórmula por trás do cosine_similarity:
v = vetor_pergunta.toarray()[0]
r0 = X[0].toarray()[0]
produto_escalar_manual = np.dot(v, r0)
sim_manual = produto_escalar_manual / (np.linalg.norm(v) * np.linalg.norm(r0) + 1e-9)
print("Exemplo manual (pergunta x 1º registro) — produto escalar:", round(produto_escalar_manual, 4),
      "| cosseno:", round(sim_manual, 4))

# vetorizado, contra todos os registros de uma vez:
similaridades = cosine_similarity(vetor_pergunta, X)[0]


# ============================================================
# 7. REGISTROS SEMELHANTES
# ============================================================

top_n = 5
indices_similares = similaridades.argsort()[::-1][:top_n]

resultado = rel.iloc[indices_similares].copy()
resultado["similaridade"] = similaridades[indices_similares]

print("\nRegistros mais semelhantes à pergunta:")
print(resultado[["Motivo do Contato", "Tema Relacionamento", "similaridade"]])


# ============================================================
# 8. IDENTIFICAÇÃO DO TEMA
# ============================================================
# Abaixo do limiar, o agente admite que não sabe em vez de
# inventar um procedimento (exigência do enunciado).

LIMIAR_CONFIANCA = 0.3

if similaridades[indices_similares[0]] < LIMIAR_CONFIANCA:
    motivo_identificado = None
    print("\nNenhum registro suficientemente semelhante encontrado.")
else:
    motivo_identificado = resultado["Motivo do Contato"].mode()[0]
    print("\nMotivo identificado:", motivo_identificado)


# ============================================================
# 9. ORIENTAÇÃO AO ESTUDANTE
# ============================================================

if motivo_identificado:
    print(f"\nOrientação: sua solicitação foi identificada como '{motivo_identificado}'. "
          f"Buscando o procedimento oficial correspondente na base de conhecimento do ASA...")
else:
    print("\nOrientação: não consegui identificar sua solicitação com confiança suficiente. "
          "Encaminhando para atendimento humano.")


# ############################################################
# ############   ENTREGA 2 - NOVOS CAMPOS   ##################
# ############################################################

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize


# ============================================================
# 10. ÁLGEBRA LINEAR AVANÇADA: SVD (ANÁLISE SEMÂNTICA LATENTE)
# ============================================================
# X ≈ U Σ Vᵀ. Os valores singulares (σ) são as raízes dos autovalores
# de XᵀX, e as linhas de Vᵀ (svd.components_) são seus autovetores.
# Mantendo só os k maiores, cada registro passa a ser um vetor de k
# "conceitos" em vez de uma dimensão por palavra. Assim, termos que
# aparecem nos mesmos contextos (ex: "boleto" e "pagamento") ficam
# próximos, mesmo sem palavras idênticas na pergunta.

# escolha de k: menor número de componentes que preserva 80% da variância
k_max = min(200, X.shape[1] - 1, X.shape[0] - 1)
svd_full = TruncatedSVD(n_components=k_max, algorithm="arpack", random_state=42).fit(X)
var_acum = svd_full.explained_variance_ratio_.cumsum()
K = max(10, int(np.argmax(var_acum >= 0.80)) + 1) if (var_acum >= 0.80).any() else k_max

svd = TruncatedSVD(n_components=K, algorithm="arpack", random_state=42)
X_red = normalize(svd.fit_transform(X))   # normalizado: produto escalar = cosseno

print("\n" + "=" * 60)
print("ENTREGA 2 - SVD")
print("=" * 60)
print(f"Dimensões: {X.shape[1]} termos -> {K} componentes "
      f"({var_acum[K-1]:.1%} da variância preservada)")
print("Matriz reduzida de registros:", X_red.shape)


# ============================================================
# 11. CONFERÊNCIA COM AUTOVALORES E AUTOVETORES
# ============================================================
# σ² da SVD deve coincidir com os maiores autovalores de XᵀX.

if X.shape[1] <= 5000:
    autovalores = np.sort(np.linalg.eigvalsh((X.T @ X).toarray()))[::-1][:3]
    print("\nσ² (SVD):           ", np.round(svd.singular_values_[:3] ** 2, 4))
    print("Autovalores de XᵀX: ", np.round(autovalores, 4))

# palavras de maior peso nos 3 primeiros componentes (autovetores):
termos = np.array(vectorizer.get_feature_names_out())
print("\nPrincipais componentes (conceitos):")
for i in range(min(3, K)):
    top = svd.components_[i].argsort()[::-1][:6]
    print(f"  Componente {i+1}:", ", ".join(termos[top]))


# ============================================================
# 12. SIMILARIDADE DE COSSENO NO ESPAÇO REDUZIDO
# ============================================================
# A mesma pergunta (já vetorizada na seção 5) é projetada no
# espaço de k conceitos e comparada com todos os registros.

vetor_pergunta_svd = normalize(svd.transform(vetor_pergunta))
similaridades_svd = cosine_similarity(vetor_pergunta_svd, X_red)[0]


# ============================================================
# 13. REGISTROS SEMELHANTES (COM SVD)
# ============================================================

indices_similares_svd = similaridades_svd.argsort()[::-1][:top_n]

resultado_svd = rel.iloc[indices_similares_svd].copy()
resultado_svd["similaridade"] = similaridades_svd[indices_similares_svd]

print("\nRegistros mais semelhantes à pergunta (com SVD):")
print(resultado_svd[["Motivo do Contato", "Tema Relacionamento", "similaridade"]])


# ============================================================
# 14. IDENTIFICAÇÃO DO TEMA (COM SVD)
# ============================================================
# No espaço reduzido as similaridades tendem a ser maiores,
# então o limiar é recalibrado (valor inicial, ajustar com testes).

LIMIAR_CONFIANCA_SVD = 0.5

if similaridades_svd[indices_similares_svd[0]] < LIMIAR_CONFIANCA_SVD:
    motivo_identificado_svd = None
    print("\nNenhum registro suficientemente semelhante encontrado (SVD).")
else:
    motivo_identificado_svd = resultado_svd["Motivo do Contato"].mode()[0]
    print("\nMotivo identificado (SVD):", motivo_identificado_svd)


# ============================================================
# 15. ORIENTAÇÃO AO ESTUDANTE (COM SVD)
# ============================================================

if motivo_identificado_svd:
    print(f"\nOrientação (SVD): sua solicitação foi identificada como '{motivo_identificado_svd}'. "
          f"Buscando o procedimento oficial correspondente na base de conhecimento do ASA...")
else:
    print("\nOrientação (SVD): não consegui identificar sua solicitação com confiança suficiente. "
          "Encaminhando para atendimento humano.")


# ============================================================
# 16. COMPARAÇÃO: TF-IDF (ENTREGA 1) x TF-IDF + SVD (ENTREGA 2)
# ============================================================

print("\n" + "=" * 60)
print("COMPARAÇÃO PARA A PERGUNTA:", pergunta)
print("=" * 60)
print(f"TF-IDF       | maior similaridade: {similaridades.max():.4f} "
      f"| limiar: {LIMIAR_CONFIANCA} | motivo: {motivo_identificado}")
print(f"TF-IDF + SVD | maior similaridade: {similaridades_svd.max():.4f} "
      f"| limiar: {LIMIAR_CONFIANCA_SVD} | motivo: {motivo_identificado_svd}")

Matriz de registros históricos: (96628, 477)
Digite a pergunta do estudante: perdi minha carteirinha preciso de outra
Exemplo manual (pergunta x 1º registro) — produto escalar: 0.0 | cosseno: 0.0

Registros mais semelhantes à pergunta:
                                  Motivo do Contato Tema Relacionamento  \
59634  Prouni: Transferência para Outra Instituição     Risco de Evasão   
3986   Prouni: Transferência para Outra Instituição     Risco de Evasão   
91322  Prouni: Transferência para Outra Instituição     Risco de Evasão   
65872  Prouni: Transferência para Outra Instituição     Risco de Evasão   
87866  Prouni: Transferência para Outra Instituição     Risco de Evasão   

       similaridade  
59634      0.594313  
3986       0.594313  
91322      0.594313  
65872      0.594313  
87866      0.594313  

Motivo identificado: Prouni: Transferência para Outra Instituição

Orientação: sua solicitação foi identificada como 'Prouni: Transferência para Outra Instituição'. Buscando o proc